# Check the modeling environment

Run this notebook in Google Colab using **Runtime > Run all**. Colab executes Python on a hosted machine, so the local Windows DLL block does not affect that runtime.

The code loads the existing 231/99 split and scaling outputs from an immutable GitHub commit. It checks the modeling packages, checks the prepared data, and fits the single-cluster baseline. It keeps the existing split and scaling.

An .ipynb file is a Jupyter notebook: explanations and executable Python cells are saved together. The code is part of the assignment analysis; the student writes and assembles the final slides.

## 1. Prepare packages and check imports

The install cell uses the runtime's existing packages when they already satisfy the requirements. It installs a missing dependency if needed. The import cell explicitly loads numpy.random, where the Windows failure occurred, and prints the installed versions.

In [ ]:
%pip install -q numpy scipy pandas scikit-learn matplotlib

In [ ]:
import sys
import numpy as np
import numpy.random
import scipy
import pandas as pd
import sklearn
import matplotlib
from sklearn.cluster import KMeans

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Ready for K-means')

## 2. Load and check the prepared data

Only the six named features enter K-means. Member IDs are retained in the tables for tracking.

`ddof=0` checks the population-standard-deviation convention used by scale_data.py. The test rows must match a transformation using the saved training means and standard deviations.

In [ ]:
DATA_REF = "24cd0a915b7a3ee05f6a4635190d05130385f25b"
BASE_URL = f"https://raw.githubusercontent.com/FrankZhao111/ClusterHW/{DATA_REF}/"
FEATURES = [
    "age",
    "visits_per_month",
    "avg_visit_minutes",
    "classes_per_month",
    "months_member",
    "monthly_extras_spend",
]

train = pd.read_csv(BASE_URL + "train_scaled.csv")
test = pd.read_csv(BASE_URL + "test_scaled.csv")
raw_train = pd.read_csv(BASE_URL + "train_members.csv")
raw_test = pd.read_csv(BASE_URL + "test_members.csv")
parameters = pd.read_csv(BASE_URL + "scaling_parameters.csv").set_index("feature").loc[FEATURES]

assert list(train.columns) == ["member_id", *FEATURES]
assert list(test.columns) == ["member_id", *FEATURES]
assert train.shape == (231, 7) and test.shape == (99, 7)
assert train["member_id"].is_unique and test["member_id"].is_unique
assert not set(train["member_id"]) & set(test["member_id"])
assert (parameters["training_count"] == 231).all()
assert (parameters["ddof"] == 0).all()

X_train = train.loc[:, FEATURES].to_numpy(dtype=float)
X_test = test.loc[:, FEATURES].to_numpy(dtype=float)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()
np.testing.assert_allclose(X_train.mean(axis=0), 0, atol=1e-10)
np.testing.assert_allclose(X_train.std(axis=0, ddof=0), 1, atol=1e-10)

# Align raw rows by member ID to check every transformed value.
for raw, scaled in [(raw_train, train), (raw_test, test)]:
    aligned = raw.set_index("member_id").loc[scaled["member_id"], FEATURES]
    expected = (aligned - parameters["training_mean"]) / parameters["training_std"]
    np.testing.assert_allclose(
        expected.to_numpy(dtype=float),
        scaled.loc[:, FEATURES].to_numpy(dtype=float),
        rtol=1e-12,
        atol=1e-12,
    )

print('Data checks passed: 231 training members, 99 test members, six features.')
print('GitHub data commit:', DATA_REF)

## 3. Check that fitting works

Fit the k=1 baseline on training members. The test spread uses this model's existing center.

`inertia_` is the training sum of squared distances. `score` returns the negative objective, so negate it to obtain a positive test spread. These are spread values, not percentages or accuracy scores.

In [ ]:
baseline = KMeans(n_clusters=1, random_state=42, n_init=20)
baseline.fit(X_train)
train_spread = float(baseline.inertia_)
test_spread = float(-baseline.score(X_test))

# Reference values were independently calculated from the verified input files.
np.testing.assert_allclose(
    [train_spread, test_spread],
    [1386.0, 666.1365956205666],
    rtol=1e-10,
    atol=1e-10,
)
print(f'k=1 training spread: {train_spread:.4f}')
print(f'k=1 test spread: {test_spread:.4f}')
print('Environment check passed: imports, data checks, and K-means fitting work.')

## Next step

After this notebook runs successfully, fit the remaining candidates k=2 through k=8 on the same training matrix, evaluate them on the existing test matrix, and compare the elbow curves. The final choice of k follows that analysis.

To keep an executed copy, use Colab's File menu to save a copy to GitHub or download the notebook and upload it to the repository. Run outputs belong with the code and analysis; write the final slides yourself.